## **Feature Importance Analysis via Permutation Importance**

### Note: Notebook 2 (ThimkersProcessing) must be run first on the same kernel.

---

### **Overview**

This notebook analyzes the directional impact of features on the remote work prediction using **sklearn's built-in `permutation_importance`** applied to the best Random Forest model from the tuning notebook.

`permutation_importance` works by shuffling one feature column at a time across multiple repeats and measuring how much the model's accuracy drops on average. A large drop means the feature is important. The built-in implementation is efficient and returns the mean and standard deviation of importance scores across all repeats.

### **Steps**
1. Restore the best RF model and test data from the processing notebook
2. Run `permutation_importance` (n_repeats=10) on the full test set
3. Rank features by mean importance drop
4. Analyze direction: for each top feature, compare the Remote rate between high vs low feature values using Mann-Whitney U
5. Produce a signed importance chart (blue = positive/remote, red = negative/anti-remote)


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import mannwhitneyu
from sklearn.inspection import permutation_importance
from scipy.stats import ttest_1samp


In [7]:
%store -r
print("Variables restored successfully.")
print(f"RF model: {rf}")
print(f"X_test shape: {X_test_scaled.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Surviving features: {len(current_features)}")


Variables restored successfully.
RF model: RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)
X_test shape: (9839, 395)
y_test shape: (9839,)
Surviving features: 198


## **Permutation Importance Computation**

Using `sklearn.inspection.permutation_importance` with `n_repeats=10` on the test set.
Each feature is shuffled 10 times; the mean and std of the accuracy drop are reported.


In [9]:
from sklearn.metrics import accuracy_score

feature_names = current_features  # 198 cultivated features from ThimkersCultivation

# Slice test set down to only the 198 surviving features
X_clean_cols = X_clean.columns.tolist()
feature_indices = [X_clean_cols.index(f) for f in feature_names]
X_test_cult = X_test_scaled[:, feature_indices]

n_features = len(feature_names)
n_repeats = 10
total_calls = n_features * n_repeats

print(f"Permutation Importance Setup:")
print(f"  Features  : {n_features}")
print(f"  Repeats   : {n_repeats}")
print(f"  Total evaluations : {total_calls}")
print(f"  Test set  : {X_test_cult.shape[0]} samples\n")

# Progress-tracking scorer
# permutation_importance calls the scorer once per (feature x repeat)
call_counter = [0]

def tracking_scorer(estimator, X, y):
    call_counter[0] += 1
    current = call_counter[0]
    feat_num = (current - 1) // n_repeats + 1
    repeat_num = (current - 1) % n_repeats + 1
    remaining = total_calls - current
    if repeat_num == 1:
        print(f"  [{current:>4}/{total_calls}] Feature {feat_num:>3}/{n_features} — {remaining} evaluations left")
    return accuracy_score(y, estimator.predict(X))

print("Starting...\n")
result = permutation_importance(
    rf,
    X_test_cult,
    y_test,
    n_repeats=n_repeats,
    random_state=42,
    n_jobs=1,          # must be 1 so the tracker runs in order
    scoring=tracking_scorer
)

perm_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Drop': result.importances_mean,
    'Std_Drop': result.importances_std
})
perm_df = perm_df.sort_values('Mean_Drop', ascending=False).reset_index(drop=True)

print(f"\nDone. {call_counter[0]} total evaluations completed.")
print(f"\nTop 5 features by permutation importance:")
print(f"{'Rank':<6} {'Feature':<45} {'Mean Drop':>10} {'Std':>8}")
print("-" * 75)
for i, row in perm_df.head(5).iterrows():
    print(f"{i+1:<6} {row['Feature']:<45} {row['Mean_Drop']:>10.4f} {row['Std_Drop']:>8.4f}")


Permutation Importance Setup:
  Features  : 198
  Repeats   : 10
  Total evaluations : 1980
  Test set  : 9839 samples

Starting...

  [   1/1980] Feature   1/198 — 1979 evaluations left


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\

  [  11/1980] Feature   2/198 — 1969 evaluations left


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\

  [  21/1980] Feature   3/198 — 1959 evaluations left


...

  [1971/1980] Feature 198/198 — 9 evaluations left


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\

  [1981/1980] Feature 199/198 — -1 evaluations left


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



Done. 1981 total evaluations completed.

Top 5 features by permutation importance:
Rank   Feature                                        Mean Drop      Std
---------------------------------------------------------------------------
1      org_size                                          0.0690   0.0023
2      work_exp                                          0.0096   0.0019
3      region_northern_america                           0.0089   0.0013
4      employment_employed                               0.0086   0.0015
5      region_eastern_europe                             0.0066   0.0010


## **Top 20 Features by Permutation Importance**


In [10]:
top20 = perm_df.head(20).sort_values('Mean_Drop')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top20['Mean_Drop'],
    y=top20['Feature'],
    orientation='h',
    error_x=dict(type='data', array=top20['Std_Drop'].tolist(), visible=True),
    marker_color='royalblue'
))
fig.update_layout(
    title='Random Forest — Top 20 Features by Permutation Importance (Mean Accuracy Drop)',
    xaxis_title='Mean Accuracy Drop',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print(f"{'Rank':<6} {'Feature':<45} {'Mean Drop':>10} {'Std':>8}")
print("-" * 75)
for i, row in perm_df.head(20).iterrows():
    print(f"{i+1:<6} {row['Feature']:<45} {row['Mean_Drop']:>10.4f} {row['Std_Drop']:>8.4f}")


Rank   Feature                                        Mean Drop      Std
---------------------------------------------------------------------------
1      org_size                                          0.0690   0.0023
2      work_exp                                          0.0096   0.0019
3      region_northern_america                           0.0089   0.0013
4      employment_employed                               0.0086   0.0015
5      region_eastern_europe                             0.0066   0.0010
6      employment_independent                            0.0058   0.0007
7      years_code                                        0.0043   0.0011
8      collab_jira                                       0.0037   0.0010
9      region_south_america                              0.0030   0.0006
10     platform_amazon_web_services_aws                  0.0024   0.0011
11     profession_professional dev                       0.0024   0.0008
12     lang_python                              

## **Statistical Significance of Importances**

For each of the top 20 features, a **one-sample t-test** is run on the 10 repeat importance scores against a null hypothesis of zero (i.e., "this feature has no effect"). A feature is considered statistically significant if **p < 0.05**, meaning the accuracy drop is unlikely to be due to chance alone.


In [ ]:
top20_names = perm_df.head(20)['Feature'].tolist()
sig_results = []

for feat in top20_names:
    feat_idx = feature_names.index(feat)
    repeat_scores = result.importances[feat_idx]  # all 10 raw drops for this feature

    t_stat, p_val = ttest_1samp(repeat_scores, popmean=0)
    mean_drop = repeat_scores.mean()
    std_drop = repeat_scores.std()

    sig_results.append({
        'Feature': feat,
        'Mean_Drop': mean_drop,
        'Std_Drop': std_drop,
        't_stat': t_stat,
        'p_value': p_val,
        'Significant': 'Yes ✓' if p_val < 0.05 else 'No ✗'
    })

sig_df = pd.DataFrame(sig_results).sort_values('Mean_Drop', ascending=False)

print(f"{'Rank':<6} {'Feature':<45} {'Mean Drop':>10} {'Std':>8} {'t-stat':>8} {'p-value':>10} {'Sig':>7}")
print("-" * 100)
for rank, (_, row) in enumerate(sig_df.iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<45} {row['Mean_Drop']:>10.4f} {row['Std_Drop']:>8.4f} {row['t_stat']:>8.3f} {row['p_value']:>10.4f} {row['Significant']:>7}")

n_sig = (sig_df['Significant'] == 'Yes ✓').sum()
print(f"\n{n_sig}/{len(sig_df)} features have statistically significant importance (p < 0.05)")


## **Direction Analysis**

For each top feature, the test set is split into **high** (value > median) vs **low** (value ≤ median) groups and the Remote class rate is compared between the two groups.

- If **Remote rate is higher in the high group** → feature is a **positive predictor** (→ Remote)
- If **Remote rate is lower in the high group** → feature is a **negative predictor** (→ Non-Remote)

A **Mann-Whitney U test** is used to check whether the difference is statistically significant (p < 0.05).


In [16]:

all_features = perm_df['Feature'].tolist()
y_test_arr = np.array(y_test)

direction_results = []

for feat in all_features:
    feat_idx = feature_names.index(feat)
    feat_vals = X_test_cult[:, feat_idx].astype(np.float64)
    median_val = np.median(feat_vals)

    high_mask = feat_vals > median_val
    low_mask  = feat_vals <= median_val

    remote_rate_high = y_test_arr[high_mask].mean() if high_mask.sum() > 0 else np.nan
    remote_rate_low  = y_test_arr[low_mask].mean()  if low_mask.sum()  > 0 else np.nan

    if high_mask.sum() > 0 and low_mask.sum() > 0:
        _, p_val = mannwhitneyu(feat_vals[y_test_arr == 1], feat_vals[y_test_arr == 0], alternative='two-sided')
    else:
        p_val = np.nan

    direction = 'Positive (→ Remote)' if remote_rate_high > remote_rate_low else 'Negative (→ Non-Remote)'
    mean_drop = perm_df.loc[perm_df['Feature'] == feat, 'Mean_Drop'].values[0]

    direction_results.append({
        'Feature': feat,
        'Mean_Drop': mean_drop,
        'Remote_Rate_High': remote_rate_high,
        'Remote_Rate_Low': remote_rate_low,
        'Direction': direction,
        'p_value': p_val,
        'Significant': 'Yes' if (not np.isnan(p_val) and p_val < 0.05) else 'No'
    })

dir_df = pd.DataFrame(direction_results)

print(f"{'Rank':<6} {'Feature':<45} {'Direction':<28} {'p-value':>10} {'Sig':>5}")
print("-" * 100)
for rank, (_, row) in enumerate(dir_df.iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<45} {row['Direction']:<28} {row['p_value']:>10.4f} {row['Significant']:>5}")

n_sig = (dir_df['Significant'] == 'Yes').sum()
n_pos_sig = ((dir_df['Significant'] == 'Yes') & (dir_df['Direction'].str.contains('Positive'))).sum()
n_neg_sig = ((dir_df['Significant'] == 'Yes') & (dir_df['Direction'].str.contains('Negative'))).sum()

print(f"\n{'='*100}")
print(f"Total features analysed : {len(dir_df)}")
print(f"Statistically significant (p < 0.05) : {n_sig} / {len(dir_df)}")
print(f"  → Positive (Remote predictor)       : {n_pos_sig}")
print(f"  → Negative (Non-Remote predictor)   : {n_neg_sig}")


Rank   Feature                                       Direction                       p-value   Sig
----------------------------------------------------------------------------------------------------
1      org_size                                      Positive (→ Remote)              0.0000   Yes
2      work_exp                                      Positive (→ Remote)              0.0000   Yes
3      region_northern_america                       Positive (→ Remote)              0.0000   Yes
4      employment_employed                           Negative (→ Non-Remote)             nan    No
5      region_eastern_europe                         Positive (→ Remote)              0.0000   Yes
6      employment_independent                        Negative (→ Non-Remote)          0.0000   Yes
7      years_code                                    Positive (→ Remote)              0.0000   Yes
8      collab_jira                                   Positive (→ Remote)              0.0000   Yes
9      r

In [15]:
# Signed importance: positive = remote predictor, negative = anti-predictor
dir_df['Signed_Importance'] = dir_df.apply(
    lambda r: r['Mean_Drop'] if 'Positive' in r['Direction'] else -r['Mean_Drop'], axis=1
)
dir_df_sorted = dir_df.sort_values('Signed_Importance')

colors = ['tomato' if v < 0 else 'royalblue' for v in dir_df_sorted['Signed_Importance']]

fig = go.Figure(go.Bar(
    x=dir_df_sorted['Signed_Importance'],
    y=dir_df_sorted['Feature'],
    orientation='h',
    marker_color=colors
))
fig.update_layout(
    title='Top 20 Features — Signed Permutation Importance<br><sup>Blue = Positive (→ Remote) | Red = Negative (→ Non-Remote)</sup>',
    xaxis_title='Signed Mean Accuracy Drop',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()
